# Unlearning Sweep — One Seed, All Fractions
### Master's Research: Machine Unlearning for Multi-Class Image Classification
**Author:** Mikołaj Hajder 264478

Each Kaggle session runs **one seed** across all fractions and methods.
At the end it writes `seed_{SEED}_results.json` — a flat list of every completed run
that `aggregate_results.py` can combine across sessions.

**Workflow per session:**
1. Set `SEED` in §0
2. Run §1 (setup)
3. Run §2 (training) — skipped automatically if checkpoints already exist
4. Run §3 (unlearning sweep) — crash-safe; re-run after a timeout to resume
5. Run §4 (collect) — writes the per-seed JSON
6. Download `seed_{SEED}_results.json` and run locally:
   ```bash
   python aggregate_results.py --results-dir checkpoints/
   ```

---
## §0  Configuration — edit `SEED` each session

In [ ]:
# ── ONE seed per session ───────────────────────────────────────────────────────
SEED      = 42                         # ← change to 43, 44, … in subsequent sessions
FRACTIONS = [0.005, 0.01,  0.05, 0.10]  # forget-set sizes as fraction of train set
DATASET   = 'cifar10'                  # 'cifar10' | 'cifar100'

# ── Which methods to run ───────────────────────────────────────────────────────
RUN_NAIVE    = True
RUN_GRAD_TAU = True
RUN_SISA     = True

# ── Paths (Kaggle defaults) ────────────────────────────────────────────────────
CKPT_DIR = '/kaggle/working/checkpoints'
DATA_DIR = '/kaggle/working/data'
REPO_DIR = '/kaggle/working/master_thesis'
REPO_URL = 'https://github.com/okejka1/master_thesis.git'

n_methods = sum([RUN_NAIVE, RUN_GRAD_TAU, RUN_SISA])
print(f'Seed       : {SEED}')
print(f'Fractions  : {FRACTIONS}')
print(f'Methods    : {n_methods}')
print(f'Total runs : {len(FRACTIONS) * n_methods}')

---
## §1  Setup

In [ ]:
import os, sys, json, shutil, subprocess, time
import numpy as np
import pandas as pd

os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(DATA_DIR,  exist_ok=True)

# ── Clone / update repo ────────────────────────────────────────────────────────
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', 'pull'], cwd=REPO_DIR, check=True)

subprocess.run(['git', 'log', '--oneline', '-3'], cwd=REPO_DIR)

# ── Install deps ───────────────────────────────────────────────────────────────
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'],
               cwd=REPO_DIR, check=True)

# ── Verify ─────────────────────────────────────────────────────────────────────
sys.path.insert(0, REPO_DIR)
import torch
print(f'PyTorch : {torch.__version__}   CUDA : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')

In [ ]:
# ── Helpers ────────────────────────────────────────────────────────────────────

def is_complete(json_path: str) -> bool:
    """True only when the JSON exists AND contains status='complete'."""
    if not os.path.exists(json_path):
        return False
    try:
        with open(json_path) as f:
            return json.load(f).get('status') == 'complete'
    except Exception:
        return False


def run_script(args, complete_sentinel=None, label=''):
    """Run args as a subprocess; skip if sentinel JSON is already complete.

    The script itself handles its own resume (training_complete → skip training,
    redo only MIA).  We only need to avoid calling it when fully done.
    """
    tag = f'  [{label}]' if label else ''
    if complete_sentinel and is_complete(complete_sentinel):
        print(f'{tag} SKIP — complete')
        return True
    print(f'{tag} RUN  {" ".join(str(a) for a in args)}')
    t0 = time.time()
    rc = subprocess.run([sys.executable] + [str(a) for a in args], cwd=REPO_DIR).returncode
    elapsed = time.time() - t0
    print(f'{tag} {"OK" if rc == 0 else f"ERROR rc={rc}"}  ({elapsed:.0f}s)')
    return rc == 0


# ── Directory helpers ──────────────────────────────────────────────────────────
def sdir():
    return os.path.join(CKPT_DIR, f'seed_{SEED}')

def fdir(frac):
    return os.path.join(sdir(), f'frac_{int(frac * 100):02d}pct')

# ── Result-path helpers (used in §3, §4, §5) ──────────────────────────────────
def naive_json(frac):
    return os.path.join(fdir(frac), f'naive_{DATASET}_results.json')

def gt_json(frac):
    return os.path.join(fdir(frac), f'grad_tau_{DATASET}_results.json')

def sisa_json(frac):
    return os.path.join(sdir(), f'sisa_unlearn_{int(frac*100):02d}pct.json')

print('Helpers defined.')

---
## §2  Training — base models for this seed

Skipped automatically if checkpoints already exist.
After this cell finishes **click Save Version** so checkpoints persist across sessions.

In [ ]:
# ── Standard ResNet-18 (used by Naive + ∇τ) ───────────────────────────────────
os.makedirs(sdir(), exist_ok=True)
base_ckpt = os.path.join(sdir(), f'resnet18_{DATASET}_best.pth')
if not os.path.exists(base_ckpt):
    run_script(
        ['train.py',
         '--config',         f'configs/{DATASET}.yaml',
         '--data-root',      DATA_DIR,
         '--checkpoint-dir', sdir(),
         '--seed',           SEED],
        label=f'train seed={SEED}',
    )
else:
    print(f'  [train seed={SEED}] SKIP — checkpoint exists')

In [ ]:
# ── SISA ensemble ─────────────────────────────────────────────────────────────
if RUN_SISA:
    sisa_meta = os.path.join(sdir(), f'sisa_{DATASET}', 'ensemble_meta.json')
    if not os.path.exists(sisa_meta):
        run_script(
            ['train_sisa.py',
             '--config',         f'configs/{DATASET}.yaml',
             '--data-root',      DATA_DIR,
             '--checkpoint-dir', sdir(),
             '--seed',           SEED],
            label=f'sisa-train seed={SEED}',
        )
    else:
        print(f'  [sisa-train seed={SEED}] SKIP — ensemble_meta.json exists')
else:
    print('SISA training skipped (RUN_SISA=False)')

---
## §3  Unlearning Sweep

Loops over every fraction.  For each:
- **Naive** writes `frac_XXpct/naive_cifar10_results.json` in two phases:
  `training_complete` (after ~4 h retrain) then `complete` (after ~5 min MIA).
  A session crash between the two phases won't repeat the retrain.
- **∇τ** and **SISA** write their JSON in one shot (both are fast).

Re-running this cell after a timeout automatically resumes — completed runs are skipped.

In [ ]:
for i, frac in enumerate(FRACTIONS, 1):
    pct_tag = f'{int(frac * 100):02d}pct'
    fd = fdir(frac)
    os.makedirs(fd, exist_ok=True)

    print(f'\n{"="*60}')
    print(f'  Fraction {frac*100:.0f}%  ({i}/{len(FRACTIONS)})  seed={SEED}')
    print(f'{"="*60}')

    # Copy base checkpoint into frac dir so unlearn scripts can find it
    base_src = os.path.join(sdir(), f'resnet18_{DATASET}_best.pth')
    base_dst = os.path.join(fd,     f'resnet18_{DATASET}_best.pth')
    if not os.path.exists(base_dst) and os.path.exists(base_src):
        shutil.copy(base_src, base_dst)

    # ── Naive Retrain ──────────────────────────────────────────────────────────
    if RUN_NAIVE:
        run_script(
            ['unlearn_naive.py',
             '--config',          f'configs/{DATASET}.yaml',
             '--data-root',       DATA_DIR,
             '--checkpoint-dir',  fd,
             '--seed',            SEED,
             '--forget-fraction', frac],
            complete_sentinel=naive_json(frac),
            label=f'naive  frac={frac}',
        )

    # ── ∇τ ────────────────────────────────────────────────────────────────────
    if RUN_GRAD_TAU:
        run_script(
            ['unlearn_grad_tau.py',
             '--config',          f'configs/{DATASET}.yaml',
             '--data-root',       DATA_DIR,
             '--checkpoint-dir',  fd,
             '--seed',            SEED,
             '--forget-fraction', frac],
            complete_sentinel=gt_json(frac),
            label=f'∇τ     frac={frac}',
        )

    # ── SISA ──────────────────────────────────────────────────────────────────
    # SISA runs from seed_dir (that's where the shard tree lives).
    # We copy the generic unlearn_results.json to a fraction-specific name.
    if RUN_SISA:
        if not is_complete(sisa_json(frac)):
            ok = run_script(
                ['unlearn_sisa.py',
                 '--config',          f'configs/{DATASET}.yaml',
                 '--data-root',       DATA_DIR,
                 '--checkpoint-dir',  sdir(),
                 '--seed',            SEED,
                 '--forget-fraction', frac],
                label=f'sisa   frac={frac}',
            )
            if ok:
                src = os.path.join(sdir(), f'sisa_{DATASET}', 'unlearn_results.json')
                if os.path.exists(src):
                    shutil.copy(src, sisa_json(frac))
                    print(f'  → {sisa_json(frac)}')
        else:
            print(f'  [sisa frac={frac}] SKIP — complete')

print('\n✓ Sweep pass done.')

---
## §4  Collect — write `seed_{SEED}_results.json`

Gathers every **complete** run for this seed into one flat JSON list.
Run this after §3 finishes (or at any point to capture partial progress).
This is the file you hand to `aggregate_results.py`.

In [ ]:
records = []
missing = []

for frac in FRACTIONS:
    for label, path_fn in [
        ('naive_retrain', naive_json),
        ('grad_tau',      gt_json),
        ('sisa',          sisa_json),
    ]:
        fp = path_fn(frac)
        if is_complete(fp):
            with open(fp) as f:
                records.append(json.load(f))
        else:
            status = None
            if os.path.exists(fp):
                try:
                    status = json.load(open(fp)).get('status', 'unknown')
                except Exception:
                    status = 'corrupt'
            missing.append({'method': label, 'frac': frac, 'status': status or 'missing'})

# Write the per-seed output file
out_path = os.path.join(CKPT_DIR, f'seed_{SEED}_results.json')
with open(out_path, 'w') as f:
    json.dump(records, f, indent=2)

print(f'Written {len(records)} complete runs → {out_path}')

if missing:
    print(f'\nIncomplete ({len(missing)} runs):')
    for m in missing:
        print(f'  frac={m["frac"]*100:.0f}%  {m["method"]:<16}  status={m["status"]}')

---
## §5  Quick inline summary (optional)

Full aggregation across seeds lives in `aggregate_results.py`.
This cell shows results for the **current seed only**.

In [ ]:
if not records:
    print('No complete records — run §3 then §4 first.')
else:
    df = pd.DataFrame([
        {
            'Method':     r['method'],
            'Forget %':   r.get('forget_fraction', float('nan')) * 100,
            'Test Acc':   r['after']['test_acc'],
            'Forget Acc': r['after']['forget_acc'],
            'Retain Acc': r['after']['retain_acc'],
            'MIA-L %':    r['after']['mia_l'] * 100,
            'MIA-E %':    r['after']['mia_e'] * 100,
            'Time (s)':   r.get('unlearn_time_s', float('nan')),
        }
        for r in records
    ])
    print(f'seed={SEED}  —  {len(df)} complete runs\n')
    try:
        from IPython.display import display
        display(df.set_index(['Method', 'Forget %']).sort_index())
    except ImportError:
        print(df.to_string())